In [2]:
import numpy as np
import itertools

# =====================================================================
# 1. ENVIRONMENT AND WORLD MODEL
# =====================================================================
class DisasterEnvironment:
    def __init__(self):
        # Locations (x, y)
        self.locations = {
            't1': (2, 8), 't2': (7, 3), 't3': (1, 5), 't4': (4, 4),
            'r1': (0, 0), 'r2': (5, 5), 'r3': (9, 9)
        }
        
        # Task Definitions (FOL: Task(t), Type(t, type), Priority(t, p), Location(t, l))
        self.tasks = {
            't1': {'type': 'rescue_victim', 'priority': 3, 'loc': self.locations['t1']},  # High = 3
            't2': {'type': 'clear_debris',  'priority': 2, 'loc': self.locations['t2']},  # Medium = 2
            't4': {'type': 'deliver_medicine','priority': 2, 'loc': self.locations['t4']}, # Medium = 2 (Added from prompt description)
            't3': {'type': 'map_building',   'priority': 1, 'loc': self.locations['t3']}   # Low = 1
        }
        
        # Robot Definitions (FOL: Robot(r), Capability(r, c))
        self.robots = {
            'r1': {'cap': 'mapping', 'speed': 3.0, 'loc': self.locations['r1']},  # Fast = 3.0
            'r2': {'cap': 'medical', 'speed': 2.0, 'loc': self.locations['r2']},  # Medium = 2.0
            'r3': {'cap': 'debris',  'speed': 1.0, 'loc': self.locations['r3']}   # Slow = 1.0
        }
        
        # Compatibility Matrix (FOL rule: Compatible(Type, Capability))
        self.compatibility = {
            ('rescue_victim', 'medical'): True,
            ('deliver_medicine', 'medical'): True, # Medical robot can also deliver medicine
            ('clear_debris', 'debris'): True,
            ('map_building', 'mapping'): True
        }

    def get_distance(self, loc1, loc2):
        return abs(loc1[0] - loc2[0]) + abs(loc1[1] - loc2[1])

# =====================================================================
# 2. ASSIGNMENT AGENT - Heuristic Search and Planning
# =====================================================================
class AssignmentAgent:
    def __init__(self, env):
        self.env = env

    def calculate_cost(self, robot_id, task_id, current_loc):
        t_loc = self.env.tasks[task_id]['loc']
        dist = self.env.get_distance(current_loc, t_loc)
        speed = self.env.robots[robot_id]['speed']
        return dist / speed

    def generate_plan(self):
        task_ids = list(self.env.tasks.keys())
        robot_ids = list(self.env.robots.keys())
        
        # Sorting tasks from highest priority to lowest priority
        sorted_tasks = sorted(task_ids, key=lambda x: self.env.tasks[x]['priority'], reverse=True)
        
        best_assignment = {}
        min_total_cost = float('inf')
        
        # Test combinations (Since m robots and n tasks, we map tasks to robots)
        # Using product to allow a robot to take multiple tasks sequentially as per project description
        for combo in itertools.product(robot_ids, repeat=len(sorted_tasks)):
            current_assignment = dict(zip(sorted_tasks, combo))
            
            valid = True
            current_cost = 0
            
            # Track the current locations of the robots for sequential constraints
            robot_locations = {r: self.env.robots[r]['loc'] for r in robot_ids}
            
            for t, r in current_assignment.items():
                t_type = self.env.tasks[t]['type']
                r_cap = self.env.robots[r]['cap']
                if (t_type, r_cap) not in self.env.compatibility:
                    valid = False
                    break
                
                # Calculate cost from the current location
                current_cost += self.calculate_cost(r, t, robot_locations[r])
                # Update robot location after completing the task
                robot_locations[r] = self.env.tasks[t]['loc']
            
            if valid and current_cost < min_total_cost:
                min_total_cost = current_cost
                best_assignment = current_assignment
                
        return best_assignment, min_total_cost

# =====================================================================
# 3. CRITIC AGENT - Logical Inference and Evaluation
# =====================================================================
class CriticAgent:
    def __init__(self, env):
        self.env = env

    def evaluate_plan(self, assignment, plan_cost):
        print("\n=== CRITIC AGENT EVALUATION REPORT ===")
        violations = 0
        
        assigned_tasks = list(assignment.keys())
        if len(assigned_tasks) == len(self.env.tasks):
            print("[APPROVED] FOL Constraint 1: Exactly one robot has been rationally assigned to all tasks.")
        else:
            print("[ERROR] FOL Constraint 1 Violation: Missing or redundant task assignment detected!")
            violations += 1
            
        for t, r in assignment.items():
            t_type = self.env.tasks[t]['type']
            r_cap = self.env.robots[r]['cap']
            if (t_type, r_cap) in self.env.compatibility:
                print(f"[APPROVED] FOL Constraint 2: Task {t} ({t_type}) -> Successfully matched with Robot {r} ({r_cap}).")
            else:
                print(f"[ERROR] FOL Constraint 2 Violation: Capability mismatch between {t} and {r}!")
                violations += 1

        task_order = list(assignment.keys())
        priority_correct = True
        for i in range(len(task_order) - 1):
            p1 = self.env.tasks[task_order[i]]['priority']
            p2 = self.env.tasks[task_order[i+1]]['priority']
            if p1 < p2:
                priority_correct = False
                
        if priority_correct:
            print("[APPROVED] FOL Constraint 3: Task ordering perfectly complies with the priority hierarchy.")
        else:
            print("[ERROR] FOL Constraint 3 Violation: Low priority task scheduled before high priority task!")
            violations += 1

        # Calculate Theoretical Optimal Cost (Lower Bound)
        optimal_theoretical_cost = 0.0
        for t, t_info in self.env.tasks.items():
            min_cost_for_t = float('inf')
            for r, r_info in self.env.robots.items():
                if (t_info['type'], r_info['cap']) in self.env.compatibility:
                    dist = self.env.get_distance(r_info['loc'], t_info['loc'])
                    cost = dist / r_info['speed']
                    if cost < min_cost_for_t:
                        min_cost_for_t = cost
            optimal_theoretical_cost += min_cost_for_t

        # Calculate Efficiency (Actual Cost / Optimal Cost)
        efficiency = plan_cost / optimal_theoretical_cost if optimal_theoretical_cost > 0 else 0
        
        print(f"\n--> Total Plan Cost (Actual Cost): {plan_cost:.2f}")
        print(f"--> Theoretical Optimal Cost (Lower Bound): {optimal_theoretical_cost:.2f}")
        print(f"--> System Operational Efficiency (Actual/Optimal): {efficiency:.2f}")
        
        if violations == 0:
            print("\n*** PROJECT RESULT: DONE! The system complies with all FOL rules and operates efficiently. ***")
        else:
            print("\n*** PROJECT RESULT: REVISIONS REQUIRED! ***")

# =====================================================================
# 4. EXECUTION SIMULATOR
# =====================================================================
if __name__ == "__main__":
    env = DisasterEnvironment()
    assigner = AssignmentAgent(env)
    best_plan, total_cost = assigner.generate_plan()
    critic = CriticAgent(env)
    critic.evaluate_plan(best_plan, total_cost)


=== CRITIC AGENT EVALUATION REPORT ===
[APPROVED] FOL Constraint 1: Exactly one robot has been rationally assigned to all tasks.
[APPROVED] FOL Constraint 2: Task t1 (rescue_victim) -> Successfully matched with Robot r2 (medical).
[APPROVED] FOL Constraint 2: Task t2 (clear_debris) -> Successfully matched with Robot r3 (debris).
[APPROVED] FOL Constraint 2: Task t4 (deliver_medicine) -> Successfully matched with Robot r2 (medical).
[APPROVED] FOL Constraint 2: Task t3 (map_building) -> Successfully matched with Robot r1 (mapping).
[APPROVED] FOL Constraint 3: Task ordering perfectly complies with the priority hierarchy.

--> Total Plan Cost (Actual Cost): 16.00
--> Theoretical Optimal Cost (Lower Bound): 14.00
--> System Operational Efficiency (Actual/Optimal): 1.14

*** PROJECT RESULT: DONE! The system complies with all FOL rules and operates efficiently. ***
